### Notebook 07 — Retention Agent


##### 1. Purpose

This notebook implements the Retention Agent for the multi-agent
customer support system.

The Retention Agent:

1. Reads its assigned task from the Coordinator execution plan.
2. Validates that required agent dependencies have completed.
3. Retrieves Prediction Agent and Vector Search Agent results
   from Shared State.
4. Builds a grounded retention context.
5. Calls an injected Retention Tool.
6. Validates and normalizes the tool response.
7. Creates a validated RetentionAgentResult.
8. Stores the result in Shared State.

The Retention Agent does not call the Prediction Agent or
Vector Search Agent directly. It consumes their previously
validated outputs from Shared State.


##### 2. Technologies Used

- Python
- Pydantic
- TypedDict shared state
- Callable tool contracts
- Dependency injection
- Agent dependency validation
- Structured agent communication
- Mock tools for unit testing
- Integration test with the Databricks Serving Endpoint and VectorSearch Endpoint


##### 3. Input

The Retention Agent receives:

- MultiAgentState
- Coordinator execution plan
- PredictionAgentResult from Shared State
- Optional VectorSearchAgentResult from Shared State
- Injected Retention Tool


##### 4. Output

The Retention Agent returns a validated RetentionAgentResult
containing:

- Customer ID
- Recommended retention action
- Reason for the recommendation
- Churn prediction label
- Prediction confidence
- Supporting customer notes
- Execution status and message


##### 5. Architecture


``` text

Coordinator Agent
        |
        ▼
Retention task and dependencies
        |
        ▼
Shared State
        |
        +-- PredictionAgentResult
        |
        +-- VectorSearchAgentResult
        |
        ▼
Retention Agent
        |
        +-- Validate dependencies
        +-- Build retention context
        +-- Call injected Retention Tool
        +-- Validate tool response
        |
        ▼
RetentionAgentResult
        |
        ▼
Shared State

```


##### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models

In [0]:
%run ./02_shared_state_and_helpers


##### 7. Imports

In [0]:
import re

from typing import Any, Callable, Dict, List, Optional
from pydantic import ValidationError
from typing import get_args


##### 8. Read Dependency Results from Shared State

###### Prediction Agent result

In [0]:
def get_prediction_agent_result(
    state: MultiAgentState,
) -> PredictionAgentResult:
    """
    Retrieve and validate the Prediction Agent result
    from Shared State.
    """

    prediction_result = (
        state["agent_results"].get(
            PREDICTION_AGENT_NAME
        )
    )

    if prediction_result is None:
        raise ValueError(
            "Prediction Agent result was not found in Shared State."
        )

    if not isinstance(
        prediction_result,
        PredictionAgentResult,
    ):
        raise TypeError(
            "Prediction Agent result has an invalid type."
        )

    return prediction_result


###### Optional Vector Search Agent result

In [0]:
def get_vector_search_agent_result(
    state: MultiAgentState,
) -> Optional[VectorSearchAgentResult]:

    vector_search_result = (
        state["agent_results"].get(
            VECTOR_SEARCH_AGENT_NAME
        )
    )

    if vector_search_result is None:
        return None

    if not isinstance(
        vector_search_result,
        VectorSearchAgentResult,
    ):
        raise TypeError(
            "Vector Search Agent result has an invalid type."
        )

    return vector_search_result


##### 9. Build Supporting Notes

In [0]:
def build_supporting_notes(
    vector_search_result: Optional[
        VectorSearchAgentResult
    ],
) -> List[str]:
    """
    Extract non-empty customer notes from the
    Vector Search Agent result.
    """

    if vector_search_result is None:
        return []

    supporting_notes = []

    for search_item in vector_search_result.results:
        note = str(
            search_item.note
        ).strip()

        if note:
            supporting_notes.append(note)

    return supporting_notes

##### 10. Build the Retention Context from Shared State

In [0]:
def build_retention_context(
    state: MultiAgentState,
) -> Dict[str, Any]:
    """
    Build the validated information passed to the
    Retention Tool from previously completed agent results.
    """

    prediction_result = get_prediction_agent_result(
        state=state
    )

    vector_search_result = (
        get_vector_search_agent_result(
            state=state
        )
    )

    if prediction_result.status != "success":
        raise ValueError(
            "Prediction Agent did not complete "
            "successfully."
        )

    if (
        vector_search_result is not None
        and vector_search_result.status != "success"
    ):
        raise ValueError(
            "Vector Search Agent result exists but "
            "did not complete successfully."
        )

    customer_id = prediction_result.customer_id

    if not customer_id:
        raise ValueError(
            "Prediction Agent result is missing "
            "customer_id."
        )

    prediction_label = (
        prediction_result.predicted_category
    )

    if prediction_label is None:
        raise ValueError(
            "Prediction Agent result is missing "
            "the prediction label."
        )

    supporting_notes = []

    if vector_search_result is not None:
        supporting_notes = [
            str(item.note).strip()
            for item in vector_search_result.results
            if str(item.note).strip()
        ]

    return {
        "customer_id": customer_id,
        "prediction_label": prediction_label,
        "prediction_confidence": (
            prediction_result.confidence
        ),
        "supporting_notes": supporting_notes,
    }

##### 11. Validate the Retention Tool Response

In [0]:
def validate_retention_tool_response(
    tool_response: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Validate and normalize the response returned by
    the injected Retention Tool.

    Parameters
    ----------
    tool_response:
        Raw dictionary returned by the Retention Tool.

    Returns
    -------
    Dict[str, Any]
        Validated retention recommendation.

    Raises
    ------
    TypeError
        If the tool response is not a dictionary.

    RuntimeError
        If the Retention Tool reports failure.

    ValueError
        If required fields are missing or the
        recommended action is unsupported.
    """

    if not isinstance(tool_response, dict):
        raise TypeError(
            "Retention Tool must return a dictionary."
        )

    status = tool_response.get(
        "status"
    )

    if status != "success":
        error_message = tool_response.get(
            "message",
            "Retention Tool execution failed.",
        )

        raise RuntimeError(
            str(error_message)
        )

    recommended_action = tool_response.get(
        "recommended_action"
    )

    if recommended_action is None:
        raise ValueError(
            "Retention Tool response is missing "
            "'recommended_action'."
        )

    recommended_action = str(
        recommended_action
    ).strip()

    allowed_retention_actions = set(
        get_args(RetentionAction)
    )

    if (
        recommended_action
        not in allowed_retention_actions
    ):
        raise ValueError(
            "Retention Tool returned an unsupported "
            "retention action: "
            f"{recommended_action!r}."
        )

    action_reason = tool_response.get(
        "action_reason",
        tool_response.get("reason"),
    )

    if action_reason is None:
        raise ValueError(
            "Retention Tool response is missing "
            "'action_reason'."
        )

    action_reason = str(
        action_reason
    ).strip()

    if not action_reason:
        raise ValueError(
            "Retention Tool returned an empty "
            "'action_reason'."
        )

    return {
        "recommended_action": recommended_action,
        "action_reason": action_reason,
        "raw_tool_response": tool_response,
    }

##### 12. Retention Tool

In [0]:
def retention_tool(
    retention_context: Dict[str, Any],
) -> Dict[str, Any]:
    """
    Recommend a retention action using the validated
    Prediction Agent and Vector Search Agent results.
    """

    customer_id = retention_context.get(
        "customer_id"
    )

    prediction_label = retention_context.get(
        "prediction_label"
    )

    prediction_confidence = retention_context.get(
        "prediction_confidence"
    )

    supporting_notes = retention_context.get(
        "supporting_notes",
        [],
    )

    if not customer_id:
        raise ValueError(
            "Retention context is missing customer_id."
        )

    if not prediction_label:
        raise ValueError(
            "Retention context is missing "
            "prediction_label."
        )

    combined_notes = " ".join(
        supporting_notes
    ).lower()

    # Customer is not predicted to churn.
    if prediction_label == "No Churn":
        return {
            "tool": "retention_tool",
            "status": "success",
            "recommended_action": "no_action",
            "action_reason": (
                "The customer is not currently "
                "predicted to churn."
            ),
        }

    # Billing or pricing concerns.
    if any(
        keyword in combined_notes
        for keyword in (
            "billing",
            "charge",
            "charges",
            "price",
            "pricing",
            "cost",
        )
    ):
        return {
            "tool": "retention_tool",
            "status": "success",
            "recommended_action": "billing_review",
            "action_reason": (
                "The customer is predicted to churn "
                "and the supporting notes indicate "
                "billing or pricing concerns."
            ),
        }

    # Service-quality concerns.
    if any(
        keyword in combined_notes
        for keyword in (
            "slow",
            "buffer",
            "internet",
            "connection",
            "connectivity",
            "service quality",
        )
    ):
        return {
            "tool": "retention_tool",
            "status": "success",
            "recommended_action": (
                "service_quality_review"
            ),
            "action_reason": (
                "The customer is predicted to churn "
                "and the supporting notes indicate "
                "service quality concerns."
            ),
        }

    # High-confidence churn prediction.
    if (
        prediction_confidence is not None
        and prediction_confidence >= 0.80
    ):
        return {
            "tool": "retention_tool",
            "status": "success",
            "recommended_action": "offer_discount",
            "action_reason": (
                "The customer has a high predicted "
                "churn risk and no specific billing "
                "or service concern was identified."
            ),
        }

    # Default proactive retention action.
    return {
        "tool": "retention_tool",
        "status": "success",
        "recommended_action": (
            "offer_support_package"
        ),
        "action_reason": (
            "The customer may benefit from "
            "additional proactive support."
        ),
    }

##### 13. Execute the Retention Agent

In [0]:
def execute_retention_agent(
    state: MultiAgentState,
    task: AgentTask,
    retention_tool: RetentionToolFunction,
) -> RetentionAgentResult:
    """
    Execute the Retention Agent's assigned task.
    """

    if task.agent_name != RETENTION_AGENT_NAME:
        raise ValueError(
            "The assigned task does not belong to "
            "retention_agent."
        )

    retention_context = build_retention_context(
        state=state
    )

    raw_tool_response = retention_tool(
        retention_context
    )

    validated_response = (
        validate_retention_tool_response(
            tool_response=raw_tool_response
        )
    )

    result_payload = {
        "agent_name": RETENTION_AGENT_NAME,
        "status": "success",
        "message": (
            "Retention Agent completed successfully "
            "and recommended "
            f"'{validated_response['recommended_action']}'."
        ),
        "task_description": task.task_description,
        "error": None,
        "task_id": task.task_id,
        "customer_id": retention_context[
            "customer_id"
        ],
        "recommended_action": (
            validated_response[
                "recommended_action"
            ]
        ),
        "action_reason": (
            validated_response[
                "action_reason"
            ]
        ),
        "prediction_label": retention_context[
            "prediction_label"
        ],
        "prediction_confidence": (
            retention_context[
                "prediction_confidence"
            ]
        ),
        "supporting_notes": retention_context[
            "supporting_notes"
        ],
    }

    return RetentionAgentResult.model_validate(
        result_payload
    )

##### 14. Run the Retention Agent

In [0]:
def run_retention_agent(
    state: MultiAgentState,
    retention_tool: RetentionToolFunction,
) -> MultiAgentState:
    """
    Run the Retention Agent inside the shared workflow.
    """

    try:
        coordinator_result = state.get(
            "coordinator_result"
        )

        if coordinator_result is None:
            raise ValueError(
                "Retention Agent cannot run because "
                "coordinator_result is missing."
            )

        assigned_task = find_assigned_agent_task(
            coordinator_result=coordinator_result,
            agent_name=RETENTION_AGENT_NAME,
        )

        if assigned_task is None:
            record_agent_execution(
                state=state,
                agent_name=RETENTION_AGENT_NAME,
                status="skipped",
                message=(
                    "Retention Agent was not required "
                    "by the execution plan."
                ),
            )

            return state

        validate_task_dependencies(
            state=state,
            task=assigned_task,
        )

        retention_result = execute_retention_agent(
            state=state,
            task=assigned_task,
            retention_tool=retention_tool,
        )

        store_agent_result(
            state=state,
            agent_result=retention_result,
        )

        record_agent_execution(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            status="success",
            message=(
                "Retention task completed successfully."
            ),
        )

    except (
        ValueError,
        TypeError,
        KeyError,
        RuntimeError,
        ValidationError,
    ) as exc:

        error_message = str(exc)

        record_agent_execution(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            status="failed",
            message=(
                "Retention Agent failed to complete "
                "the assigned task."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            error_code="RETENTION_AGENT_ERROR",
            error_message=error_message,
        )

    except Exception as exc:

        error_message = (
            "Unexpected Retention Agent error: "
            f"{exc}"
        )

        record_agent_execution(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            status="failed",
            message=(
                "Retention Agent encountered an "
                "unexpected error."
            ),
        )

        record_agent_error(
            state=state,
            agent_name=RETENTION_AGENT_NAME,
            error_code=(
                "RETENTION_AGENT_UNEXPECTED_ERROR"
            ),
            error_message=error_message,
        )

    return state

##### 15. Tests

In [0]:
def test_retention_agent_unit_test() -> None:
    """
    Run deterministic unit tests for the
    Retention Agent.

    Tests:
    1. Successful retention recommendation.
    2. Successful no-action recommendation.
    3. Retention Agent skip behavior.
    4. Missing Prediction Agent dependency.
    5. Retention Tool failure.
    6. Invalid retention action.
    7. Missing action reason.
    8. Successful recommendation when prediction
       confidence is unavailable.
    """

    # =========================================================
    # Shared test helpers
    # =========================================================

    def create_mock_prediction_result(
        customer_id: str = "7590-VHVEG",
        prediction_label: str = "Churn",
        confidence: Optional[float] = 0.91,
    ) -> PredictionAgentResult:
        """
        Create a successful Prediction Agent result.
        """

        return PredictionAgentResult(
            agent_name=PREDICTION_AGENT_NAME,
            status="success",
            message=(
                "Prediction Agent completed successfully."
            ),
            task_description=(
                f"Predict churn for customer "
                f"{customer_id}."
            ),
            error=None,
            customer_id=customer_id,
            predicted_category=prediction_label,
            confidence=confidence,
            model_name="mock_churn_model",
            raw_prediction=(
                prediction_label == "Churn"
            ),
        )

    def create_mock_vector_search_result(
        customer_id: str = "7590-VHVEG",
    ) -> VectorSearchAgentResult:
        """
        Create a successful Vector Search Agent result.
        """

        return VectorSearchAgentResult(
            agent_name=VECTOR_SEARCH_AGENT_NAME,
            status="success",
            message=(
                "Vector Search Agent retrieved "
                "supporting customer notes."
            ),
            task_description=(
                "Find customer concerns related "
                "to churn."
            ),
            error=None,
            task_id="task_2",
            query=(
                "Find customer concerns related "
                "to churn."
            ),
            results=[
                VectorSearchItem(
                    customer_id=customer_id,
                    note=(
                        "Customer complained about "
                        "high monthly charges."
                    ),
                    similarity_score=0.94,
                ),
                VectorSearchItem(
                    customer_id=customer_id,
                    note=(
                        "Customer asked about cancelling "
                        "the service."
                    ),
                    similarity_score=0.88,
                ),
            ],
        )

    def create_retention_coordinator_result(
        include_retention_task: bool = True,
        depends_on: Optional[List[AgentName]] = None,
    ) -> CoordinatorResult:
        """
        Create a Coordinator result for
        Retention Agent unit tests.
        """

        execution_plan = []

        if include_retention_task:
            execution_plan.append(
                AgentTask(
                    task_id="task_3",
                    agent_name=RETENTION_AGENT_NAME,
                    task_description=(
                        "Recommend a retention action for "
                        "customer 7590-VHVEG."
                    ),
                    depends_on=(
                        depends_on
                        if depends_on is not None
                        else [
                            PREDICTION_AGENT_NAME,
                            VECTOR_SEARCH_AGENT_NAME,
                        ]
                    ),
                )
            )
        else:
            execution_plan.append(
                AgentTask(
                    task_id="task_1",
                    agent_name=SQL_AGENT_NAME,
                    task_description=(
                        "Count churned customers."
                    ),
                    depends_on=[],
                )
            )

        return CoordinatorResult(
            agent_name=COORDINATOR_AGENT_NAME,
            status="success",
            message=(
                "Execution plan created successfully."
            ),
            task_description=(
                "Plan the multi-agent workflow."
            ),
            error=None,
            request_type=(
                "retention"
                if include_retention_task
                else "sql_analytics"
            ),
            reasoning=(
                "The workflow was created for "
                "unit testing."
            ),
            execution_plan=execution_plan,
        )

    # =========================================================
    # Mock Retention Tools
    # =========================================================

    def mock_retention_tool(
        retention_context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        Return a deterministic rule-based
        retention recommendation.
        """

        prediction_label = retention_context[
            "prediction_label"
        ]

        prediction_confidence = (
            retention_context.get(
                "prediction_confidence"
            )
        )

        supporting_notes = retention_context.get(
            "supporting_notes",
            [],
        )

        combined_notes = " ".join(
            supporting_notes
        ).lower()

        if prediction_label == "No Churn":
            return {
                "status": "success",
                "recommended_action": "no_action",
                "action_reason": (
                    "The customer is not currently "
                    "predicted to churn."
                ),
            }

        if any(
            keyword in combined_notes
            for keyword in (
                "billing",
                "charge",
                "pricing",
                "cost",
            )
        ):
            return {
                "status": "success",
                "recommended_action": "billing_review",
                "action_reason": (
                    "The customer is predicted to churn "
                    "and the supporting notes indicate "
                    "billing or pricing concerns."
                ),
            }

        if any(
            keyword in combined_notes
            for keyword in (
                "slow",
                "buffer",
                "internet",
                "connection",
                "connectivity",
            )
        ):
            return {
                "status": "success",
                "recommended_action": (
                    "service_quality_review"
                ),
                "action_reason": (
                    "The customer is predicted to churn "
                    "and the supporting notes indicate "
                    "service quality concerns."
                ),
            }

        if (
            prediction_confidence is not None
            and prediction_confidence >= 0.80
        ):
            return {
                "status": "success",
                "recommended_action": (
                    "offer_discount"
                ),
                "action_reason": (
                    "The customer has a high predicted "
                    "churn risk and no specific complaint "
                    "was identified."
                ),
            }

        return {
            "status": "success",
            "recommended_action": (
                "offer_support_package"
            ),
            "action_reason": (
                "The customer may benefit from "
                "additional proactive support."
            ),
        }

    def mock_retention_tool_failure(
        retention_context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        Simulate Retention Tool failure.
        """

        return {
            "status": "error",
            "message": (
                "Mock Retention Tool execution failed."
            ),
        }

    def mock_retention_tool_invalid_action(
        retention_context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        Return an unsupported retention action.
        """

        return {
            "status": "success",
            "recommended_action": "free_phone",
            "action_reason": (
                "This action is not supported."
            ),
        }

    def mock_retention_tool_missing_reason(
        retention_context: Dict[str, Any],
    ) -> Dict[str, Any]:
        """
        Return a response with the required
        action reason intentionally omitted.
        """

        return {
            "status": "success",
            "recommended_action": "offer_discount",
        }

    # =========================================================
    # TEST 1: Successful recommendation
    # =========================================================

    print("=" * 80)
    print("TEST 1: Successful retention recommendation")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][RETENTION_AGENT_NAME]

    assert isinstance(
        result,
        RetentionAgentResult,
    )

    assert result.status == "success"

    assert (
        result.customer_id
        == "7590-VHVEG"
    )

    assert (
        result.recommended_action
        == "billing_review"
    )

    assert result.prediction_label == "Churn"

    assert result.prediction_confidence == 0.91

    assert len(
        result.supporting_notes
    ) == 2

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 2: Successful no-action recommendation
    # =========================================================

    print("=" * 80)
    print("TEST 2: Successful no-action recommendation")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 5575-GNVDE."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result(
            depends_on=[
                PREDICTION_AGENT_NAME,
            ]
        )
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result(
        customer_id="5575-GNVDE",
        prediction_label="No Churn",
        confidence=0.84,
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    result = updated_state[
        "agent_results"
    ][RETENTION_AGENT_NAME]

    assert result.status == "success"

    assert (
        result.recommended_action
        == "no_action"
    )

    assert (
        result.prediction_label
        == "No Churn"
    )

    assert result.supporting_notes == []

    assert updated_state["errors"] == []

    print("PASS")
    print(result.model_dump())
    print()

    # =========================================================
    # TEST 3: Retention Agent skips
    # =========================================================

    print("=" * 80)
    print("TEST 3: Retention Agent skips")
    print("=" * 80)

    state = create_initial_state(
        "How many customers churned?"
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result(
            include_retention_task=False
        )
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "skipped"
    )

    print("PASS")
    print(
        updated_state[
            "execution_history"
        ][-1]
    )
    print()

    # =========================================================
    # TEST 4: Missing Prediction Agent dependency
    # =========================================================

    print("=" * 80)
    print("TEST 4: Missing Prediction Agent dependency")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "prediction"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 5: Retention Tool failure
    # =========================================================

    print("=" * 80)
    print("TEST 5: Retention Tool failure")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=(
            mock_retention_tool_failure
        ),
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "execution failed"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 6: Invalid retention action
    # =========================================================

    print("=" * 80)
    print("TEST 6: Invalid retention action")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=(
            mock_retention_tool_invalid_action
        ),
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "unsupported"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 7: Missing action reason
    # =========================================================

    print("=" * 80)
    print("TEST 7: Missing action reason")
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result()
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result()

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = create_mock_vector_search_result()

    updated_state = run_retention_agent(
        state=state,
        retention_tool=(
            mock_retention_tool_missing_reason
        ),
    )

    assert (
        RETENTION_AGENT_NAME
        not in updated_state["agent_results"]
    )

    assert len(
        updated_state["errors"]
    ) == 1

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "failed"
    )

    assert (
        "action_reason"
        in updated_state[
            "errors"
        ][-1].error_message.lower()
    )

    print("PASS")
    print(updated_state["errors"][-1])
    print()

    # =========================================================
    # TEST 8: Confidence is unavailable
    # =========================================================

    print("=" * 80)
    print(
        "TEST 8: Successful recommendation with "
        "confidence=None"
    )
    print("=" * 80)

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        create_retention_coordinator_result(
            depends_on=[
                PREDICTION_AGENT_NAME,
            ]
        )
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = create_mock_prediction_result(
        confidence=None,
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=mock_retention_tool,
    )

    assert (
        RETENTION_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][RETENTION_AGENT_NAME]

    assert result.status == "success"

    assert (
        result.prediction_confidence
        is None
    )

    assert (
        result.recommended_action
        == "offer_support_package"
    )

    assert updated_state["errors"] == []

    print("PASS")
    print(result.model_dump())
    print()

    print("=" * 80)
    print(
        "ALL RETENTION AGENT UNIT TESTS PASSED"
    )
    print("=" * 80)

In [0]:
def test_retention_agent_integration_test() -> None:
    """
    Test the Retention Agent with the real
    rule-based Retention Tool.
    """

    retention_task = AgentTask(
        task_id="task_3",
        agent_name=RETENTION_AGENT_NAME,
        task_description=(
            "Recommend a retention action for "
            "customer 7590-VHVEG."
        ),
        depends_on=[
            PREDICTION_AGENT_NAME,
            VECTOR_SEARCH_AGENT_NAME,
        ],
    )

    final_task = AgentTask(
        task_id="task_4",
        agent_name=FINAL_RESPONSE_AGENT_NAME,
        task_description=(
            "Generate the final grounded response."
        ),
        depends_on=[
            RETENTION_AGENT_NAME,
        ],
    )

    coordinator_result = CoordinatorResult(
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        request_type="retention",
        reasoning=(
            "Retention requires prediction and "
            "supporting customer information."
        ),
        execution_plan=[
            retention_task,
            final_task,
        ],
    )

    state = create_initial_state(
        "Recommend a retention action for "
        "customer 7590-VHVEG."
    )

    state["coordinator_result"] = (
        coordinator_result
    )

    state["agent_results"][
        PREDICTION_AGENT_NAME
    ] = PredictionAgentResult(
        agent_name=PREDICTION_AGENT_NAME,
        status="success",
        message="Prediction completed.",
        customer_id="7590-VHVEG",
        predicted_category="Churn",
        confidence=None,
        raw_prediction=True,
    )

    state["agent_results"][
        VECTOR_SEARCH_AGENT_NAME
    ] = VectorSearchAgentResult(
        agent_name=VECTOR_SEARCH_AGENT_NAME,
        status="success",
        message="Vector search completed.",
        task_id="task_2",
        query="Find customer retention concerns.",
        results=[
            VectorSearchItem(
                customer_id="7590-VHVEG",
                note=(
                    "Customer reported repeated "
                    "connectivity problems."
                ),
                similarity_score=0.90,
            )
        ],
    )

    updated_state = run_retention_agent(
        state=state,
        retention_tool=retention_tool,
    )

    print("Retention Agent Result:")
    print(
        updated_state[
            "agent_results"
        ].get(RETENTION_AGENT_NAME)
    )

    print("\nExecution History:")
    print(
        updated_state["execution_history"]
    )

    print("\nErrors:")
    print(
        updated_state["errors"]
    )

    assert (
        RETENTION_AGENT_NAME
        in updated_state["agent_results"]
    )

    result = updated_state[
        "agent_results"
    ][RETENTION_AGENT_NAME]

    assert result.status == "success"

    assert result.recommended_action is not None

    assert result.action_reason

    assert updated_state["errors"] == []

    assert (
        updated_state[
            "execution_history"
        ][-1].status
        == "success"
    )

    print(
        "Retention Agent integration test passed."
    )

##### 16. Key Learnings

1. The Retention Agent does not directly call other agents.

2. It consumes validated PredictionAgentResult and VectorSearchAgentResult objects from Shared State.

3. Coordinator task dependencies determine which agent results must exist before the Retention Agent can execute.

4. The Retention Agent builds a focused retention context rather than passing the entire Shared State to the Retention Tool.

5. The Retention Tool is injected through a Callable contract.

6. Mock and rule-based tools support inexpensive, deterministic learning and testing.

7. The Retention Tool can later be replaced with an LLM-backed implementation without rewriting the Retention Agent.

8. Raw tool responses are manually validated and normalized.

9. RetentionAgentResult provides the trusted Pydantic contract stored in Shared State for downstream agents.

10. This notebook demonstrates real collaboration between specialist agents through Shared State.

##### 17. Conclusion

- The Retention Agent was successfully implemented as a dependent specialist agent.

- Unlike the SQL, Prediction, and Vector Search Agents, the Retention Agent consumes outputs produced by other agents.

- It retrieves validated PredictionAgentResult and VectorSearchAgentResult objects from Shared State, builds a grounded retention context, calls an injected Retention Tool,
validates the returned recommendation, and stores a validated RetentionAgentResult.

- This demonstrates how specialist agents collaborate without calling or tightly coupling themselves to one another.

##### 18. Next Notebook

Notebook 08 will implement the Final Response Agent.

The Final Response Agent will:

- Read validated results from Shared State.
- Determine which agent results are relevant.
- Build a grounded response context.
- Generate a clear user-facing answer.
- Avoid introducing information not supported by agent results.
- Return a validated FinalResponseAgentResult.